# Compare neural operators for Poisson problem

Run in **`neuralopv2`** after training DeepONet, PCANet, and FNO.

This notebook:
1. Loads the true Poisson model and NOP surrogates via `nn_util.load_surrogate_model`
2. **In-distribution:** evaluates all test samples in latent and physical space
3. **Out-of-distribution:** samples a new prior and compares surrogate vs true forward model
4. Saves stats and sample plots under `compare_nops/in_distribution/` and `compare_nops/out_of_distribution/`


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from dolfinx import mesh
from dolfinx.fem import functionspace
from dolfinx.mesh import CellType
from mpi4py import MPI
from mpl_toolkits.axes_grid1 import ImageGrid

NOTEBOOK_DIR = os.getcwd()
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", "..", ".."))
POISSON_DIR = os.path.join(ROOT, "survey_work/problems/poisson")
MODEL_PATH = os.path.join(POISSON_DIR, "")

sys.path.insert(0, os.path.join(ROOT, "src/plotting"))
sys.path.insert(0, os.path.join(ROOT, "src/data"))
sys.path.insert(0, os.path.join(ROOT, "src/prior"))
sys.path.insert(0, os.path.join(ROOT, "src/mcmc"))
sys.path.insert(0, os.path.join(ROOT, "src/nn"))
sys.path.insert(0, os.path.join(ROOT, "src/nn/deeponet"))
sys.path.insert(0, os.path.join(ROOT, "src/nn/mlp"))
sys.path.insert(0, os.path.join(ROOT, "src/nn/pcanet"))
sys.path.insert(0, os.path.join(ROOT, "src/nn/fno"))
sys.path.insert(0, POISSON_DIR)

from field_plot import quick_field_plot, quick_field_plot_grid
from poissonModel import PoissonModel
from priorSampler import PriorSampler
from nn_util import load_surrogate_model

plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

seed = 0
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
# --- configuration ---
DATA_PREFIX = "Poisson"
NOP_NAMES = ("DeepONet", "PCANet", "FNO")
U_COMPS = 1

# Prior used to generate training / test data (in-distribution)
prior_ac = 0.005
prior_cc = 0.2
prior_logn_scale = 1.0
prior_logn_translate = 0.0

# OOD prior (change these to explore generalization)
prior_ac_ood = 0.02
prior_cc_ood = 0.5
n_ood_samples = 100

# Test indices for sample plots (indices into each NOP test set)
i_choices = [27, 47, 43, 80, 5]

# Output directories
analysis_root = os.path.join(NOTEBOOK_DIR, "in_distribution")
ood_root = os.path.join(
    NOTEBOOK_DIR,
    "out_of_distribution",
    f"prior_ac_{prior_ac_ood:g}_cc_{prior_cc_ood:g}",
)
for d in (
    os.path.join(analysis_root, "stats"),
    os.path.join(analysis_root, "plots", "samples"),
    os.path.join(ood_root, "stats"),
    os.path.join(ood_root, "plots", "samples"),
):
    os.makedirs(d, exist_ok=True)


## True Poisson model and NOP surrogates

In [ ]:
nx, ny = 50, 50
fe_order = 1

domain = mesh.create_unit_square(MPI.COMM_WORLD, nx, ny, cell_type=CellType.triangle)
Vm = functionspace(domain, ("Lagrange", fe_order))
Vu = Vm

prior_sampler = PriorSampler(Vm, prior_ac, prior_cc, seed)
model = PoissonModel(
    Vm, Vu, prior_sampler, prior_logn_scale, prior_logn_translate, seed
)
nodes = model.m_nodes

surrogates = {}
for name in NOP_NAMES:
    sm = load_surrogate_model(name, DATA_PREFIX, MODEL_PATH, model, u_comps=U_COMPS)
    if sm is not None:
        surrogates[name] = sm
    else:
        print(f"{name} surrogate not loaded")

if not surrogates:
    raise RuntimeError("No surrogate models loaded")


## Helpers

In [ ]:
def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def rel_l2_rows(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    num = np.linalg.norm(a - b, axis=1)
    den = np.linalg.norm(a, axis=1)
    return num / (den + 1e-30)


def predict_latent(name, sm):
    data, nn = sm.data, sm.model
    if name == "DeepONet":
        return to_numpy(nn.predict(data.X_test, data.X_trunk))
    return to_numpy(nn.predict(data.X_test))


def decode_u_physical(name, sm, y_latent_row):
    data = sm.data
    y = y_latent_row
    if y.ndim == 1:
        y = y.reshape(1, -1)
    u = data.decoder_Y(y)
    if name == "FNO":
        u = u[0, :, :, 0]
        u[data.u_grid_dirichlet_boundary_nodes[:, 0], data.u_grid_dirichlet_boundary_nodes[:, 1]] = 0.0
        return u
    u = np.asarray(u).reshape(-1)
    u[data.u_mesh_dirichlet_boundary_nodes] = 0.0
    return u


def physical_errors(name, sm, y_pred):
    data = sm.data
    y_true = to_numpy(data.Y_test)
    n = y_true.shape[0]
    errs = np.zeros(n)
    for i in range(n):
        u_true = decode_u_physical(name, sm, y_true[i])
        u_pred = decode_u_physical(name, sm, y_pred[i])
        errs[i] = np.linalg.norm(u_true - u_pred) / (np.linalg.norm(u_true) + 1e-30)
    return errs


def save_field_plot(name, sm, u, path, cmap="jet"):
    if name == "FNO":
        grid_x = sm.data.grid_x_test[0, :, :, 0]
        grid_y = sm.data.grid_y_test[0, :, :, 0]
        quick_field_plot_grid(u, grid_x, grid_y, cmap=cmap, savefilename=path, show_plot=False)
    else:
        quick_field_plot(u, nodes, cmap=cmap, savefilename=path, show_plot=False)
    plt.close()


## 1. In-distribution evaluation (test set)

In [ ]:
in_dist_stats = {}

for name, sm in surrogates.items():
    print(f"\n=== {name} in-distribution ===")
    y_pred = predict_latent(name, sm)
    y_true = to_numpy(sm.data.Y_test)

    err_latent = rel_l2_rows(y_true, y_pred)
    err_physical = physical_errors(name, sm, y_pred)

    in_dist_stats[name] = {
        "latent": err_latent,
        "physical": err_physical,
    }

    print(
        f"Latent   rel L2: mean={err_latent.mean():.3e}, std={err_latent.std():.3e}"
    )
    print(
        f"Physical rel L2: mean={err_physical.mean():.3e}, std={err_physical.std():.3e}"
    )

stats_dir = os.path.join(analysis_root, "stats")
np.savez(
    os.path.join(stats_dir, "errors.npz"),
    **{f"{k}_latent": v["latent"] for k, v in in_dist_stats.items()},
    **{f"{k}_physical": v["physical"] for k, v in in_dist_stats.items()},
)

with open(os.path.join(stats_dir, "summary.txt"), "w") as f:
    for name, d in in_dist_stats.items():
        f.write(
            f"{name}: latent mean={d['latent'].mean():.6e} std={d['latent'].std():.6e}\n"
        )
        f.write(
            f"{name}: physical mean={d['physical'].mean():.6e} std={d['physical'].std():.6e}\n"
        )


### Sample plots (in-distribution)

In [ ]:
plot_dir = os.path.join(analysis_root, "plots", "samples")
cmaps = ["jet", "jet", "hot"]
plt.ioff()

# Use DeepONet data for shared m / true u on mesh (all NOPs share same test split)
ref_name = "DeepONet" if "DeepONet" in surrogates else next(iter(surrogates))
ref_sm = surrogates[ref_name]
y_true_ref = to_numpy(ref_sm.data.Y_test)

for i in i_choices:
    print(f"i = {i}")

    i_m = ref_sm.data.decoder_X(to_numpy(ref_sm.data.X_test[i]))
    save_field_plot(ref_name, ref_sm, np.asarray(i_m).reshape(-1), f"{plot_dir}/m_{i}.png", cmaps[0])

    i_u = decode_u_physical(ref_name, ref_sm, y_true_ref[i])
    if ref_name == "FNO":
        save_field_plot(ref_name, ref_sm, i_u, f"{plot_dir}/u_{i}.png", cmaps[1])
    else:
        save_field_plot(ref_name, ref_sm, i_u, f"{plot_dir}/u_{i}.png", cmaps[1])

    for name, sm in surrogates.items():
        y_pred = predict_latent(name, sm)
        i_u_pred = decode_u_physical(name, sm, y_pred[i])
        save_field_plot(name, sm, i_u_pred, f"{plot_dir}/u_{name.lower()}_{i}.png", cmaps[1])

        if name == "FNO":
            i_u_true_grid = decode_u_physical(name, sm, to_numpy(sm.data.Y_test[i]))
            err_field = i_u_true_grid - i_u_pred
            save_field_plot(name, sm, err_field, f"{plot_dir}/u_{name.lower()}_error_{i}.png", cmaps[2])
        else:
            err_field = i_u - i_u_pred
            save_field_plot(name, sm, err_field, f"{plot_dir}/u_{name.lower()}_error_{i}.png", cmaps[2])

        err_norm = in_dist_stats[name]["physical"][i]
        np.savetxt(f"{plot_dir}/u_{name.lower()}_error_norm_{i}.txt", [err_norm])


### Montage (in-distribution)

In [ ]:
montage_path = os.path.join(analysis_root, "plots", "nops_comparison_montage.png")
fig = plt.figure(figsize=(16, 16))
grid = ImageGrid(fig, 111, nrows_ncols=(len(i_choices), 5), axes_pad=0.1)

lbls = ["m", "True", "DeepONet", "PCANet", "FNO"]
imgs = []
for i in i_choices:
    imgs.append(plt.imread(f"{plot_dir}/m_{i}.png"))
    imgs.append(plt.imread(f"{plot_dir}/u_{i}.png"))
    for nop in ("DeepONet", "PCANet", "FNO"):
        path = f"{plot_dir}/u_{nop.lower()}_{i}.png"
        imgs.append(plt.imread(path) if os.path.isfile(path) else np.zeros((10, 10, 3)))

for ax, im in zip(grid, imgs):
    ax.imshow(im)
    ax.axis("off")

for j, lbl in enumerate(lbls):
    grid[j].set_title(lbl)

fig.savefig(montage_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved {montage_path}")


## 2. Out-of-distribution evaluation (new prior)

In [ ]:
prior_ood = PriorSampler(Vm, prior_ac_ood, prior_cc_ood, seed + 1)

ood_stats = {name: np.zeros(n_ood_samples) for name in surrogates}

print(f"OOD prior: ac={prior_ac_ood}, cc={prior_cc_ood}, n={n_ood_samples}")
t0 = time.time()

for k in range(n_ood_samples):
    w = prior_ood()[0]
    m = model.transform_gaussian_pointwise(w)
    u_true = model.solveFwd(u=None, m=m, transform_m=False)

    for name, sm in surrogates.items():
        u_pred = sm.solveFwd(w)
        ood_stats[name][k] = np.linalg.norm(u_true - u_pred) / (np.linalg.norm(u_true) + 1e-30)

    if (k + 1) % max(1, n_ood_samples // 10) == 0:
        print(f"  OOD sample {k + 1}/{n_ood_samples}")

print(f"OOD finished in {time.time() - t0:.1f}s")

ood_stats_dir = os.path.join(ood_root, "stats")
np.savez(os.path.join(ood_stats_dir, "errors.npz"), **ood_stats)

with open(os.path.join(ood_stats_dir, "summary.txt"), "w") as f:
    f.write(f"prior_ac={prior_ac_ood}, prior_cc={prior_cc_ood}, n={n_ood_samples}\n")
    for name, errs in ood_stats.items():
        f.write(f"{name}: mean={errs.mean():.6e} std={errs.std():.6e}\n")
        print(f"{name} OOD rel L2: mean={errs.mean():.3e}, std={errs.std():.3e}")


### Sample plots (OOD)

In [ ]:
ood_plot_dir = os.path.join(ood_root, "plots", "samples")
n_ood_plots = min(5, n_ood_samples)
ood_plot_indices = list(range(n_ood_plots))

for j in ood_plot_indices:
    w = prior_ood()[0]
    m = model.transform_gaussian_pointwise(w)
    u_true = model.solveFwd(u=None, m=m, transform_m=False)

    quick_field_plot(m, nodes, cmap="jet", savefilename=f"{ood_plot_dir}/m_ood_{j}.png", show_plot=False)
    plt.close()
    quick_field_plot(u_true, nodes, cmap="jet", savefilename=f"{ood_plot_dir}/u_true_ood_{j}.png", show_plot=False)
    plt.close()

    for name, sm in surrogates.items():
        u_pred = sm.solveFwd(w)
        quick_field_plot(u_pred, nodes, cmap="jet", savefilename=f"{ood_plot_dir}/u_{name.lower()}_ood_{j}.png", show_plot=False)
        plt.close()
        err = u_true - u_pred
        quick_field_plot(err, nodes, cmap="hot", savefilename=f"{ood_plot_dir}/u_{name.lower()}_error_ood_{j}.png", show_plot=False)
        plt.close()

print(f"OOD sample plots saved to {ood_plot_dir}")
